## Contexto del avance:
En este primer avance se trabajó en la preparación inicial del proyecto, con foco en la generación, organización y exploración de los datos necesarios para el análisis posterior.  
El objetivo principal fue construir una base de trabajo confiable que permita desarrollar las siguientes etapas del proyecto de manera ordenada.

Durante esta etapa se definieron los datos a utilizar, se realizaron procesos de generación/carga de información y se verificó que la estructura obtenida fuera consistente con los requerimientos planteados.  
Este avance sirve como punto de partida para validar que los datos disponibles sean suficientes, coherentes y útiles para las próximas fases.


## Desarrollo: 
En el desarrollo de este avance se llevaron a cabo distintas tareas relacionadas con la construcción del entorno de trabajo y la preparación de los datos.  
Primero, se implementó el proceso de generación de datos mediante scripts específicos, asegurando que la información creada respondiera a las variables y criterios definidos para el proyecto.

Luego, se organizó la información obtenida y se realizó una revisión general de su contenido para identificar posibles errores, valores faltantes o inconsistencias.  
Además, se documentaron los pasos realizados con el fin de mantener trazabilidad sobre el proceso y facilitar futuras correcciones o mejoras.

Finalmente, se dejó preparada una base inicial sobre la cual será posible aplicar análisis, transformaciones y validaciones más profundas en los próximos avances.

Asimismo en el se crearon en DBEABER una conexion con Postgres donde se crearon las columans pertienents para setear el data set

### Justificación de tablas

Las tablas implementadas fueron definidas para representar las entidades principales del sistema logístico y sus relaciones:

- `vehicles`: almacena la información de los vehículos disponibles, incluyendo tipo, capacidad y estado operativo.
- `drivers`: registra los conductores asignables a los viajes.
- `routes`: contiene las rutas entre ciudades, con distancia estimada y duración prevista.
- `trips`: representa cada viaje realizado, vinculando vehículo, conductor y ruta.
- `deliveries`: modela las entregas individuales asociadas a cada viaje.
- `maintenance`: registra eventos de mantenimiento vinculados a los vehículos.

Esta estructura permite modelar un flujo logístico completo, donde un vehículo y un conductor realizan un viaje sobre una ruta, y dicho viaje puede contener múltiples entregas. A su vez, los vehículos requieren seguimiento de mantenimiento, por lo que esa tabla complementa el análisis operativo.

### Explicación de `generate_trips()` y `_get_hourly_distribution()`

El método `generate_trips()` fue diseñado para generar masivamente viajes distribuidos a lo largo de dos años de operación. Su funcionamiento se basa en los siguientes pasos:

- Obtiene primero los IDs válidos de vehículos activos, conductores activos y rutas existentes.
- Define una fecha inicial ubicada dos años atrás.
- Para cada viaje, selecciona aleatoriamente un vehículo, un conductor y una ruta.
- Calcula la hora de salida usando una distribución horaria no uniforme, para simular mayor actividad en horario laboral.
- Estima una duración real del viaje a partir de la duración prevista de la ruta, incorporando una variación aleatoria.
- Calcula la fecha de llegada, el consumo de combustible y el peso transportado.
- Define el estado del viaje: si la llegada ya ocurrió, se registra como `completed`; si no, como `in_progress`.
- Inserta los viajes en lotes para mejorar el rendimiento de carga en la base de datos.

El método auxiliar `_get_hourly_distribution()` devuelve un arreglo de probabilidades para las 24 horas del día. Su propósito es evitar una generación totalmente uniforme y hacer que los viajes se concentren en franjas horarias más realistas:

- baja actividad durante la madrugada,
- mayor actividad entre las 6:00 y las 20:00,
- picos más altos entre las 8:00 y las 12:00,
- y otro bloque de alta actividad entre las 14:00 y las 18:00.

Gracias a esta función, la generación de viajes resulta más cercana a un comportamiento operativo real.


### ERROR EN TYPE AL CARGAR LOS DATOS
En la función generate_deliveries(), en la línea 389 del archivo 
[A1-01_data_generation_estudiantes.py] tuve un error en la instrucción:

package_weight = weights[i]
El problema aparecía porque el valor de weights[i] no se estaba generando como un float nativo 
de Python, sino como un np.float64, que es un tipo numérico propio de la librería NumPy. 
Al momento de insertar ese dato en PostgreSQL, el adaptador psycopg2 no lo interpretó correctamente,
 y por eso se produjo el error.

Averiguando el problema, entendí que la columna de la base de datos sí acepta valores decimales,
 por lo tanto el inconveniente no estaba en PostgreSQL ni en la definición de la tabla, 
 sino en el tipo de dato que el script estaba enviando desde Python.

La solución fue convertir explícitamente ese valor a float antes de insertarlo, quedando así:


package_weight = float(weights[i])
Elegí esa solución porque psycopg2 trabaja correctamente con tipos nativos de Python, 
como float, pero puede presentar problemas con tipos numéricos de NumPy como np.float64.



## Resultados y validación:
Como resultado de este avance, se obtuvo un conjunto de datos inicial correctamente generado y estructurado, listo para ser utilizado en las siguientes etapas. 
Se verificó que los archivos producidos pudieran cargarse sin errores y que la información contuviera el formato esperado.

Las funciones que intervienen en la validación y control de calidad de los datos son principalmente validate_data_quality(), generate_trips(), generate_deliveries() y _distribute_weight().

La validación principal se concentra en validate_data_quality(), donde se comprueba la integridad referencial entre tablas, la consistencia temporal de los viajes, el control de peso respecto de la capacidad del vehículo y la presencia de tracking number en las entregas.

Además, durante la generación de datos, funciones como generate_trips() y generate_deliveries() aplican reglas de consistencia para evitar registros incoherentes. Por su parte, _distribute_weight() contribuye a mantener valores realistas en la distribución del peso de los paquetes.


Entonces la validación se realizó mediante revisiones generales de los datos, comprobando:
- cantidad de registros generados,
- presencia de campos obligatorios,
- formato correcto de los valores,
- ausencia de errores evidentes en la estructura.

Estas verificaciones permitieron confirmar que el avance cumple con su objetivo inicial y que la base construida es adecuada para continuar con el desarrollo del proyecto.



## Conclusiones del avance:

En conclusión, el avance 1 permitió establecer una base sólida para el proyecto, especialmente en lo relacionado con la generación y preparación de los datos.  
Se logró organizar un primer conjunto de información confiable, junto con un proceso de trabajo claro y reproducible.

Además, esta etapa permitió detectar la importancia de validar los datos desde el inicio, ya que esto reduce errores en fases posteriores y mejora la calidad general del proyecto.  
Como siguiente paso, será posible profundizar en el análisis, procesamiento o modelado de la información, partiendo de una estructura ya verificada.